# Full 114M-Name SapBERT Fine-Tuning V3: Mined Hard Negatives

V3 is the follow-up experiment to run if V2 does not clearly improve external-lab retrieval. Instead of only using random or lexically bucketed in-batch negatives, this notebook mines hard negatives from the current embedding space: names that the model thinks are very similar but belong to different CIDs.

The goal is to improve top-1 ranking and MRR by teaching the model to separate confusing near-neighbor chemical names while preserving synonym closeness.

Main outputs:
- `results/sapbert_full_114m_hard_negative_V3/artifacts/model_best`
- `results/sapbert_full_114m_hard_negative_V3/artifacts/model_last`
- `results/sapbert_full_114m_hard_negative_V3/artifacts/mined_hard_triplets.parquet`
- validation history and plots under `results/sapbert_full_114m_hard_negative_V3/`


## Install Dependencies

Run this first in Colab. It avoids forcing a PyTorch reinstall, but installs FAISS for hard-negative mining.


In [ ]:
import subprocess
import sys

base_packages = [
    'pandas==2.2.2',
    'numpy>=1.26.4,<2.1',
    'pyarrow>=15.0',
    'duckdb>=1.0',
    'tqdm>=4.66',
    'PyYAML>=6.0',
    'transformers>=4.40',
    'accelerate>=0.28',
    'sentencepiece>=0.2',
    'matplotlib>=3.8',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *base_packages])

try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-gpu-cu12'])
except subprocess.CalledProcessError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu>=1.8'])

print('Installed notebook dependencies.')


## Mount Drive and Locate Repo


In [ ]:
from pathlib import Path
import os
import shutil
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)


def find_repo_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/content/drive/MyDrive/ChemSynonymizer/chem_synonymizer'),
        Path('/content/drive/MyDrive/chem_synonymizer'),
        Path('/content/chem_synonymizer'),
    ]
    for candidate in candidates:
        if (candidate / 'src' / 'chem_synonymizer').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not find repo root containing src/chem_synonymizer. Update find_repo_dir().')


REPO_DIR = find_repo_dir()
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo:', REPO_DIR)


## Configuration

`START_MODEL_MODE = 'auto_external'` chooses V2 only if its external-lab metric beats the earlier clean SapBERT contrastive model. If the V2 external result is missing or worse, V3 starts from the older clean checkpoint, which has been the stronger model so far.


In [ ]:
import gc
import json
import math
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

from chem_synonymizer.embed import encode_texts_to_memmap, pool_outputs
from chem_synonymizer.utils import save_json, set_seed

RUN_NAME = 'sapbert_full_114m_hard_negative_V3'
RESULTS_RUN_DIR = REPO_DIR / 'results' / RUN_NAME
RESULTS_ARTIFACTS_DIR = RESULTS_RUN_DIR / 'artifacts'
MODEL_BEST_DIR = RESULTS_ARTIFACTS_DIR / 'model_best'
MODEL_LAST_DIR = RESULTS_ARTIFACTS_DIR / 'model_last'
CHECKPOINT_ROOT = RESULTS_ARTIFACTS_DIR / 'checkpoints'
PLOTS_DIR = RESULTS_RUN_DIR / 'plots'
HISTORY_CSV_PATH = RESULTS_RUN_DIR / 'v3_hard_negative_history.csv'

SCRATCH_ROOT = Path('/mnt/local-scratch') if Path('/mnt/local-scratch').exists() else Path('/content')
WORK_DIR = SCRATCH_ROOT / 'chem_synonymizer' / RUN_NAME
WORK_ARTIFACTS_DIR = WORK_DIR / 'artifacts'

V2_RESULTS_DIR = REPO_DIR / 'results' / 'sapbert_full_114m_contrastive_V2'
V2_ARTIFACTS_DIR = V2_RESULTS_DIR / 'artifacts'
V2_TRAIN_GROUPS_DRIVE = V2_ARTIFACTS_DIR / 'train_cid_groups_v2.parquet'
V2_VAL_GROUPS_DRIVE = V2_ARTIFACTS_DIR / 'val_cid_groups_v2.parquet'
V2_GROUP_STATS_DRIVE = V2_ARTIFACTS_DIR / 'group_stats_v2.json'

TRAIN_GROUPS_PATH = WORK_ARTIFACTS_DIR / 'train_cid_groups_v2.parquet'
VAL_GROUPS_PATH = WORK_ARTIFACTS_DIR / 'val_cid_groups_v2.parquet'
GROUP_STATS_PATH = WORK_ARTIFACTS_DIR / 'group_stats_v2.json'

MINING_ROWS_PATH = WORK_ARTIFACTS_DIR / 'hard_mining_rows.parquet'
MINING_EMBEDDINGS_PATH = WORK_ARTIFACTS_DIR / 'hard_mining_anchor_embeddings.npy'
MINED_TRIPLETS_PATH = RESULTS_ARTIFACTS_DIR / 'mined_hard_triplets.parquet'
MINING_META_PATH = RESULTS_ARTIFACTS_DIR / 'hard_negative_mining_meta.json'

CLEAN_MODEL_DIR = REPO_DIR / 'results' / 'sapbert_contrastive' / 'artifacts' / 'model'
V2_BEST_TRAINED_MODEL_DIR = V2_ARTIFACTS_DIR / 'model_best_trained'
SAPBERT_BASE_MODEL = 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext'

CLEAN_EXTERNAL_METRICS_CANDIDATES = [
    REPO_DIR / 'results' / 'sapbert_contrastive' / 'external_lab_unique_surface' / 'external_lab_metrics.json',
    REPO_DIR / 'results' / 'sapbert_contrastive' / 'external_lab_metrics.json',
]
V2_EXTERNAL_METRICS_CANDIDATES = [
    V2_RESULTS_DIR / 'external_lab_unique_surface_best_trained' / 'external_lab_metrics.json',
    V2_RESULTS_DIR / 'external_lab_unique_surface' / 'external_lab_metrics.json',
]

START_MODEL_MODE = 'auto_external'  # auto_external, clean, v2_best_trained, base, or a model path string.
SELECTION_METRIC = 'MRR'            # Change to HitRate@10 if you prefer candidate-generation quality.

SEED = 0
MAX_LENGTH = 96
POOLING = 'cls'
MIXED_PRECISION = 'bf16'

# Mining. Increase MINING_GROUP_LIMIT for stronger mining; 500k is a good first A100 run.
REBUILD_MINING_ROWS = False
REMINE_HARD_NEGATIVES = False
MINING_GROUP_LIMIT = 500_000
MINING_ENCODE_BATCH_SIZE = 8192
MINING_SEARCH_BATCH_SIZE = 8192
MINING_SEARCH_K = 64
USE_GPU_FAISS = True
USE_FLOAT16_FAISS = True

# Training. Each step encodes 3 * TRAIN_TRIPLET_BATCH_SIZE strings.
TRAIN_TRIPLET_BATCH_SIZE = 128
MAX_TRAIN_STEPS = 5_000
LEARNING_RATE = 5e-6
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_GRAD_NORM = 1.0
TEMPERATURE = 0.05
TRIPLET_MARGIN = 0.05
HARD_NEGATIVE_LOSS_WEIGHT = 0.5
SAVE_EVERY_STEPS = 1000
VALIDATE_EVERY_STEPS = 250
VALIDATION_PAIR_LIMIT = 8192
VALIDATION_ENCODE_BATCH_SIZE = 512
VALIDATION_SCORE_BATCH_SIZE = 512
KEEP_LAST_N_CHECKPOINTS = 4
OVERWRITE_MODEL = False
RESUME_FROM_CHECKPOINT = True

for path in [RESULTS_RUN_DIR, RESULTS_ARTIFACTS_DIR, CHECKPOINT_ROOT, PLOTS_DIR, WORK_ARTIFACTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
print('Results:', RESULTS_RUN_DIR)
print('Scratch:', WORK_DIR, shutil.disk_usage(SCRATCH_ROOT))


## Helpers and Start Checkpoint Selection


In [ ]:
def load_json(path: Path) -> dict:
    with Path(path).open('r', encoding='utf-8') as f:
        return json.load(f)


def first_existing(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


def metric_from_candidates(paths: list[Path], metric: str) -> tuple[float | None, Path | None]:
    path = first_existing(paths)
    if path is None:
        return None, None
    payload = load_json(path)
    value = payload.get(metric)
    if value is None:
        value = payload.get(metric.replace('@', 'At'))
    return (float(value) if value is not None else None), path


def choose_start_model() -> tuple[str, dict]:
    mode = str(START_MODEL_MODE)
    clean_exists = (CLEAN_MODEL_DIR / 'config.json').exists()
    v2_exists = (V2_BEST_TRAINED_MODEL_DIR / 'config.json').exists()
    details = {'mode': mode, 'selection_metric': SELECTION_METRIC}

    if mode == 'clean':
        return (str(CLEAN_MODEL_DIR) if clean_exists else SAPBERT_BASE_MODEL), details
    if mode == 'v2_best_trained':
        return (str(V2_BEST_TRAINED_MODEL_DIR) if v2_exists else str(CLEAN_MODEL_DIR)), details
    if mode == 'base':
        return SAPBERT_BASE_MODEL, details
    if mode not in {'auto_external', 'clean', 'v2_best_trained', 'base'}:
        return mode, details

    clean_metric, clean_path = metric_from_candidates(CLEAN_EXTERNAL_METRICS_CANDIDATES, SELECTION_METRIC)
    v2_metric, v2_path = metric_from_candidates(V2_EXTERNAL_METRICS_CANDIDATES, SELECTION_METRIC)
    details.update({
        'clean_metric': clean_metric,
        'clean_metrics_path': str(clean_path) if clean_path else None,
        'v2_metric': v2_metric,
        'v2_metrics_path': str(v2_path) if v2_path else None,
    })

    if v2_exists and v2_metric is not None and (clean_metric is None or v2_metric >= clean_metric):
        details['chosen'] = 'v2_best_trained'
        return str(V2_BEST_TRAINED_MODEL_DIR), details
    if clean_exists:
        details['chosen'] = 'clean'
        return str(CLEAN_MODEL_DIR), details
    if v2_exists:
        details['chosen'] = 'v2_best_trained_fallback'
        return str(V2_BEST_TRAINED_MODEL_DIR), details
    details['chosen'] = 'sapbert_base_fallback'
    return SAPBERT_BASE_MODEL, details


START_MODEL_SOURCE, start_model_details = choose_start_model()
print('Start model:', START_MODEL_SOURCE)
print(json.dumps(start_model_details, indent=2))


def copy_if_needed(src: Path, dst: Path) -> None:
    if dst.exists() and src.exists() and dst.stat().st_size == src.stat().st_size:
        return
    if not src.exists():
        raise FileNotFoundError(
            f'Missing required V2 artifact: {src}. Run the V2 notebook through the group-building stage first.'
        )
    dst.parent.mkdir(parents=True, exist_ok=True)
    print(f'Restoring {src.name} to scratch...')
    shutil.copy2(src, dst)


def restore_v2_group_artifacts() -> None:
    copy_if_needed(V2_TRAIN_GROUPS_DRIVE, TRAIN_GROUPS_PATH)
    copy_if_needed(V2_VAL_GROUPS_DRIVE, VAL_GROUPS_PATH)
    copy_if_needed(V2_GROUP_STATS_DRIVE, GROUP_STATS_PATH)
    print('Restored V2 group artifacts.')
    print(json.dumps(load_json(GROUP_STATS_PATH), indent=2)[:3000])


def save_config() -> None:
    payload = {
        'run_name': RUN_NAME,
        'start_model_source': START_MODEL_SOURCE,
        'start_model_details': start_model_details,
        'seed': SEED,
        'max_length': MAX_LENGTH,
        'pooling': POOLING,
        'mixed_precision': MIXED_PRECISION,
        'mining_group_limit': MINING_GROUP_LIMIT,
        'mining_search_k': MINING_SEARCH_K,
        'train_triplet_batch_size': TRAIN_TRIPLET_BATCH_SIZE,
        'max_train_steps': MAX_TRAIN_STEPS,
        'learning_rate': LEARNING_RATE,
        'temperature': TEMPERATURE,
        'triplet_margin': TRIPLET_MARGIN,
        'hard_negative_loss_weight': HARD_NEGATIVE_LOSS_WEIGHT,
        'validate_every_steps': VALIDATE_EVERY_STEPS,
        'save_every_steps': SAVE_EVERY_STEPS,
    }
    save_json(payload, RESULTS_RUN_DIR / 'v3_config.json')


restore_v2_group_artifacts()
save_config()


## Build Mining Rows

This samples CID groups from the V2 full-114M group table and creates one anchor/positive pair per selected CID. Hard negatives are mined from these anchors in the next cell.


In [ ]:
def q(path: Path) -> str:
    return str(path).replace("'", "''")


def build_mining_rows() -> pd.DataFrame:
    if MINING_ROWS_PATH.exists() and not REBUILD_MINING_ROWS:
        df = pd.read_parquet(MINING_ROWS_PATH)
        print('Using cached mining rows:', df.shape)
        return df

    con = duckdb.connect()
    con.execute('PRAGMA preserve_insertion_order=false')
    print('Sampling CID groups for hard-negative mining...')
    selected = con.execute(f'''
        SELECT cid, names
        FROM read_parquet('{q(TRAIN_GROUPS_PATH)}')
        WHERE array_length(names) >= 2
        ORDER BY hash(cid || '|v3mine|' || CAST({int(SEED)} AS VARCHAR))
        LIMIT {int(MINING_GROUP_LIMIT)}
    ''').df()
    con.close()

    rng = np.random.default_rng(int(SEED))
    rows = []
    for row_i, row in tqdm(selected.iterrows(), total=len(selected), desc='Choosing anchor/positive pairs'):
        names = [str(name) for name in list(row['names']) if str(name)]
        if len(names) < 2:
            continue
        a_i, p_i = rng.choice(len(names), size=2, replace=False)
        rows.append({
            'row_id': len(rows),
            'cid': str(row['cid']),
            'anchor': names[int(a_i)],
            'positive': names[int(p_i)],
        })
    df = pd.DataFrame(rows)
    df.to_parquet(MINING_ROWS_PATH, index=False)
    print('Saved mining rows:', MINING_ROWS_PATH, df.shape)
    return df


mining_df = build_mining_rows()
mining_df.head()


## Mine Embedding Hard Negatives

This embeds the mining anchors with the chosen start model, builds a flat cosine FAISS index over the anchors, and assigns each anchor a nearest neighbor from a different CID as its hard negative.


In [ ]:
import faiss
import torch
from transformers import AutoModel, AutoTokenizer


def faiss_gpu_available() -> bool:
    return bool(USE_GPU_FAISS) and torch.cuda.is_available() and hasattr(faiss, 'StandardGpuResources')


def mining_meta() -> dict:
    return {
        'start_model_source': START_MODEL_SOURCE,
        'n_mining_rows': int(len(mining_df)),
        'max_length': int(MAX_LENGTH),
        'pooling': POOLING,
        'mining_search_k': int(MINING_SEARCH_K),
        'seed': int(SEED),
    }


def hard_triplets_cache_is_valid() -> bool:
    if REMINE_HARD_NEGATIVES:
        return False
    if not MINED_TRIPLETS_PATH.exists() or not MINING_META_PATH.exists():
        return False
    try:
        return load_json(MINING_META_PATH) == mining_meta()
    except Exception:
        return False


def build_flat_index(xb_memmap: np.ndarray):
    cpu_index = faiss.IndexFlatIP(int(xb_memmap.shape[1]))
    index = cpu_index
    resources = None
    if faiss_gpu_available():
        resources = faiss.StandardGpuResources()
        opts = faiss.GpuClonerOptions()
        opts.useFloat16 = bool(USE_FLOAT16_FAISS)
        index = faiss.index_cpu_to_gpu(resources, 0, cpu_index, opts)
        print('FAISS mining index: GPU flat IP')
    else:
        print('FAISS mining index: CPU flat IP')

    for start in tqdm(range(0, len(xb_memmap), int(MINING_SEARCH_BATCH_SIZE)), desc='FAISS add mining anchors'):
        end = min(start + int(MINING_SEARCH_BATCH_SIZE), len(xb_memmap))
        xb = np.asarray(xb_memmap[start:end], dtype=np.float32)
        index.add(np.ascontiguousarray(xb))
    return index, resources


def mine_hard_negatives() -> pd.DataFrame:
    if hard_triplets_cache_is_valid():
        df = pd.read_parquet(MINED_TRIPLETS_PATH)
        print('Using cached mined hard triplets:', df.shape)
        return df

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Mining embedding device:', device)
    tokenizer = AutoTokenizer.from_pretrained(START_MODEL_SOURCE, use_fast=True)
    model = AutoModel.from_pretrained(START_MODEL_SOURCE).to(device).eval()

    anchor_embeddings = encode_texts_to_memmap(
        mining_df['anchor'].astype(str).tolist(),
        out_path=MINING_EMBEDDINGS_PATH,
        tokenizer=tokenizer,
        model=model,
        model_name=START_MODEL_SOURCE,
        pooling=POOLING,
        batch_size=MINING_ENCODE_BATCH_SIZE,
        max_length=MAX_LENGTH,
        device=device,
        normalize=True,
        out_dtype='float16',
        run_key=f'{RUN_NAME}:hard_mining:{len(mining_df)}',
    )

    del model, tokenizer
    gc.collect()
    if device.startswith('cuda'):
        torch.cuda.empty_cache()

    index, resources = build_flat_index(anchor_embeddings)
    cids = mining_df['cid'].astype(str).to_numpy()
    anchors = mining_df['anchor'].astype(str).to_numpy()
    positives = mining_df['positive'].astype(str).to_numpy()
    rows = []
    rng = np.random.default_rng(int(SEED) + 13)

    for start in tqdm(range(0, len(anchor_embeddings), int(MINING_SEARCH_BATCH_SIZE)), desc='Mining nearest different-CID negatives'):
        end = min(start + int(MINING_SEARCH_BATCH_SIZE), len(anchor_embeddings))
        xq = np.asarray(anchor_embeddings[start:end], dtype=np.float32)
        scores, neighbors = index.search(np.ascontiguousarray(xq), int(MINING_SEARCH_K))
        for local_i in range(end - start):
            row_i = start + local_i
            anchor_cid = cids[row_i]
            chosen_j = None
            chosen_score = None
            for score, j in zip(scores[local_i], neighbors[local_i]):
                j = int(j)
                if j < 0 or j == row_i:
                    continue
                if cids[j] != anchor_cid and anchors[j] != anchors[row_i]:
                    chosen_j = j
                    chosen_score = float(score)
                    break
            if chosen_j is None:
                for _ in range(100):
                    j = int(rng.integers(0, len(cids)))
                    if cids[j] != anchor_cid:
                        chosen_j = j
                        chosen_score = None
                        break
            if chosen_j is None:
                continue
            rows.append({
                'anchor_cid': anchor_cid,
                'negative_cid': cids[chosen_j],
                'anchor': anchors[row_i],
                'positive': positives[row_i],
                'hard_negative': anchors[chosen_j],
                'hard_negative_score': chosen_score,
            })

    triplets = pd.DataFrame(rows)
    triplets.to_parquet(MINED_TRIPLETS_PATH, index=False)
    save_json(mining_meta(), MINING_META_PATH)
    print('Saved mined hard triplets:', MINED_TRIPLETS_PATH, triplets.shape)
    del index, resources
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return triplets


triplets_df = mine_hard_negatives()
triplets_df.head()


## Training and Validation Functions


In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup


def latest_checkpoint() -> Path | None:
    checkpoints = []
    for path in CHECKPOINT_ROOT.glob('checkpoint-*'):
        try:
            step = int(path.name.split('-')[-1])
        except ValueError:
            continue
        if (path / 'config.json').exists():
            checkpoints.append((step, path))
    return sorted(checkpoints)[-1][1] if checkpoints else None


def checkpoint_step(path: Path | None) -> int:
    if path is None:
        return 0
    try:
        return int(path.name.split('-')[-1])
    except ValueError:
        return 0


def cleanup_old_checkpoints() -> None:
    checkpoints = []
    for path in CHECKPOINT_ROOT.glob('checkpoint-*'):
        try:
            step = int(path.name.split('-')[-1])
        except ValueError:
            continue
        checkpoints.append((step, path))
    for _, path in sorted(checkpoints)[:-int(KEEP_LAST_N_CHECKPOINTS)]:
        shutil.rmtree(path, ignore_errors=True)


def load_training_history() -> list[dict]:
    if not HISTORY_CSV_PATH.exists():
        return []
    return pd.read_csv(HISTORY_CSV_PATH).to_dict('records')


def write_training_history(history: list[dict]) -> None:
    if not history:
        return
    df = pd.DataFrame(history)
    dedupe_cols = [col for col in ['step', 'validation_MRR', 'loss'] if col in df.columns]
    if dedupe_cols:
        df = df.drop_duplicates(subset=dedupe_cols, keep='last')
    df.to_csv(HISTORY_CSV_PATH, index=False)


def set_scheduler_resume_step(scheduler, optimizer, completed_steps: int) -> None:
    completed_steps = int(completed_steps)
    if completed_steps <= 0:
        return
    scheduler.last_epoch = completed_steps
    if not hasattr(scheduler, 'lr_lambdas'):
        return
    lrs = [
        float(base_lr) * float(lr_lambda(completed_steps))
        for base_lr, lr_lambda in zip(scheduler.base_lrs, scheduler.lr_lambdas)
    ]
    for group, lr in zip(optimizer.param_groups, lrs):
        group['lr'] = lr
    scheduler._last_lr = lrs


def supervised_contrastive_loss(embeddings: torch.Tensor, labels: torch.Tensor, temperature: float) -> torch.Tensor:
    labels = labels.to(embeddings.device)
    sim = torch.matmul(embeddings, embeddings.T) / float(temperature)
    sim = sim - sim.max(dim=1, keepdim=True).values.detach()
    eye = torch.eye(labels.numel(), dtype=torch.bool, device=embeddings.device)
    positive_mask = labels[:, None].eq(labels[None, :]) & ~eye
    logits_mask = ~eye
    exp_logits = torch.exp(sim) * logits_mask.float()
    log_prob = sim - torch.log(exp_logits.sum(dim=1, keepdim=True).clamp_min(1e-12))
    positives_per_anchor = positive_mask.sum(dim=1)
    valid = positives_per_anchor > 0
    mean_log_prob = (positive_mask.float() * log_prob).sum(dim=1) / positives_per_anchor.clamp_min(1)
    return -mean_log_prob[valid].mean()


def hard_negative_triplet_loss(embeddings: torch.Tensor, triplet_count: int, margin: float) -> torch.Tensor:
    anchors = embeddings[:triplet_count]
    positives = embeddings[triplet_count:2 * triplet_count]
    negatives = embeddings[2 * triplet_count:3 * triplet_count]
    pos_sim = (anchors * positives).sum(dim=1)
    neg_sim = (anchors * negatives).sum(dim=1)
    return torch.relu(neg_sim - pos_sim + float(margin)).mean()


def encode_train_batch(texts: list[str], tokenizer, model, device: str) -> torch.Tensor:
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=int(MAX_LENGTH),
        return_tensors='pt',
    ).to(device)
    if MIXED_PRECISION == 'bf16':
        amp_dtype = torch.bfloat16
        amp_enabled = device.startswith('cuda')
    elif MIXED_PRECISION == 'fp16':
        amp_dtype = torch.float16
        amp_enabled = device.startswith('cuda')
    else:
        amp_dtype = torch.float32
        amp_enabled = False
    with torch.autocast(device_type='cuda' if device.startswith('cuda') else 'cpu', dtype=amp_dtype, enabled=amp_enabled):
        out = model(**enc)
        emb = pool_outputs(out.last_hidden_state, enc['attention_mask'], POOLING)
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
    del enc, out
    return emb


def hard_triplet_batch_iterator(triplets: pd.DataFrame, batch_size: int, seed: int, skip_batches: int = 0):
    rng = np.random.default_rng(int(seed) + int(skip_batches))
    n = len(triplets)
    if n == 0:
        raise ValueError('No mined hard triplets available.')
    arrays = {
        'anchor': triplets['anchor'].astype(str).to_numpy(),
        'positive': triplets['positive'].astype(str).to_numpy(),
        'hard_negative': triplets['hard_negative'].astype(str).to_numpy(),
    }
    seen = 0
    while True:
        order = rng.permutation(n)
        for start in range(0, n, int(batch_size)):
            if seen < skip_batches:
                seen += 1
                continue
            idx = order[start:start + int(batch_size)]
            if len(idx) == 0:
                continue
            anchors = arrays['anchor'][idx].tolist()
            positives = arrays['positive'][idx].tolist()
            negatives = arrays['hard_negative'][idx].tolist()
            labels = list(range(len(idx))) + list(range(len(idx))) + list(range(len(idx), 2 * len(idx)))
            yield anchors + positives + negatives, torch.tensor(labels, dtype=torch.long), int(len(idx))
            seen += 1


def validation_pairs_from_groups(limit: int, seed: int) -> tuple[list[str], list[str]]:
    table = pq.read_table(VAL_GROUPS_PATH, columns=['names']).slice(0, int(limit))
    df = table.to_pandas()
    rng = np.random.default_rng(int(seed))
    anchors = []
    positives = []
    for names in df['names'].tolist():
        names = [str(name) for name in list(names) if str(name)]
        if len(names) < 2:
            continue
        a_i, p_i = rng.choice(len(names), size=2, replace=False)
        anchors.append(names[int(a_i)])
        positives.append(names[int(p_i)])
    return anchors, positives


def encode_texts_for_eval(texts: list[str], tokenizer, model, device: str, batch_size: int) -> torch.Tensor:
    chunks = []
    was_training = model.training
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(texts), int(batch_size)):
            batch = texts[start:start + int(batch_size)]
            emb = encode_train_batch(batch, tokenizer, model, device)
            chunks.append(emb.detach().cpu())
    if was_training:
        model.train()
    return torch.cat(chunks, dim=0)


def evaluate_validation_retrieval(tokenizer, model, device: str, limit: int = VALIDATION_PAIR_LIMIT) -> dict:
    anchors, positives = validation_pairs_from_groups(limit, seed=SEED)
    n = len(anchors)
    if n == 0:
        raise ValueError('No validation pairs available.')
    anchor_emb = encode_texts_for_eval(anchors, tokenizer, model, device, VALIDATION_ENCODE_BATCH_SIZE).to(device)
    positive_emb = encode_texts_for_eval(positives, tokenizer, model, device, VALIDATION_ENCODE_BATCH_SIZE).to(device)
    ranks = []
    with torch.inference_mode():
        for start in range(0, n, int(VALIDATION_SCORE_BATCH_SIZE)):
            end = min(start + int(VALIDATION_SCORE_BATCH_SIZE), n)
            scores = anchor_emb[start:end] @ positive_emb.T
            target = scores[torch.arange(end - start, device=device), torch.arange(start, end, device=device)]
            rank = (scores > target[:, None]).sum(dim=1) + 1
            ranks.append(rank.detach().cpu())
            del scores, target, rank
    ranks = torch.cat(ranks).numpy()
    payload = {
        'n_pairs': int(n),
        'MRR': float(np.mean(1.0 / ranks)),
        'HitRate@1': float(np.mean(ranks <= 1)),
        'HitRate@5': float(np.mean(ranks <= 5)),
        'HitRate@10': float(np.mean(ranks <= 10)),
        'median_rank': float(np.median(ranks)),
    }
    del anchor_emb, positive_emb
    if device.startswith('cuda'):
        torch.cuda.empty_cache()
    return payload


def save_training_checkpoint(model, tokenizer, step: int, history: list[dict]) -> None:
    ckpt_dir = CHECKPOINT_ROOT / f'checkpoint-{int(step):06d}'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(ckpt_dir)
    tokenizer.save_pretrained(ckpt_dir)
    save_json({'step': int(step), 'history_tail': history[-10:]}, ckpt_dir / 'training_state.json')
    write_training_history(history)
    cleanup_old_checkpoints()


def save_best_model(model, tokenizer, step: int, metrics: dict) -> None:
    MODEL_BEST_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(MODEL_BEST_DIR)
    tokenizer.save_pretrained(MODEL_BEST_DIR)
    save_json({'best_step': int(step), 'validation_metrics': metrics}, RESULTS_RUN_DIR / 'best_validation_metrics.json')


## Fine-Tune with Mined Hard Negatives

This objective combines supervised contrastive loss over anchor/positive pairs with a margin loss that explicitly pushes the mined hard negative below the positive.


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device.startswith('cuda'):
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
print('Training device:', device)

target_steps = int(MAX_TRAIN_STEPS)
resume_checkpoint = latest_checkpoint() if RESUME_FROM_CHECKPOINT else None
resume_step = checkpoint_step(resume_checkpoint)

if (resume_step >= target_steps) and (MODEL_BEST_DIR / 'config.json').exists() and not OVERWRITE_MODEL:
    print(f'Training already reached {resume_step} steps; skipping training. Increase MAX_TRAIN_STEPS to continue.')
else:
    start_step = 0
    model_source = START_MODEL_SOURCE
    ckpt = resume_checkpoint
    if ckpt is not None and not OVERWRITE_MODEL:
        model_source = str(ckpt)
        start_step = checkpoint_step(ckpt)
        print('Resuming model weights from checkpoint:', ckpt)
    else:
        print('Starting from:', START_MODEL_SOURCE)

    tokenizer = AutoTokenizer.from_pretrained(model_source, use_fast=True)
    model = AutoModel.from_pretrained(model_source).to(device)
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=float(LEARNING_RATE), weight_decay=float(WEIGHT_DECAY))
    total_steps = int(MAX_TRAIN_STEPS)
    warmup_steps = int(total_steps * float(WARMUP_RATIO))
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    set_scheduler_resume_step(scheduler, optimizer, start_step)

    history = load_training_history() if start_step > 0 and not OVERWRITE_MODEL else []
    running_loss = None
    running_supcon = None
    running_triplet = None
    iterator = hard_triplet_batch_iterator(
        triplets_df,
        TRAIN_TRIPLET_BATCH_SIZE,
        seed=SEED,
        skip_batches=start_step,
    )
    t0 = time.perf_counter()

    initial_metrics = evaluate_validation_retrieval(tokenizer, model, device)
    best_mrr = initial_metrics['MRR']
    best_step = start_step
    if OVERWRITE_MODEL or not (MODEL_BEST_DIR / 'config.json').exists():
        save_best_model(model, tokenizer, start_step, initial_metrics)
    else:
        best_payload = load_json(RESULTS_RUN_DIR / 'best_validation_metrics.json') if (RESULTS_RUN_DIR / 'best_validation_metrics.json').exists() else {}
        old_metrics = best_payload.get('validation_metrics', {})
        old_mrr = old_metrics.get('MRR')
        if old_mrr is not None:
            best_mrr = float(old_mrr)
            best_step = best_payload.get('best_step', start_step)
    print('Initial validation:', json.dumps(initial_metrics, indent=2))
    print(f'Best validation MRR to beat: {best_mrr:.4f} at step {best_step}')

    progress = tqdm(range(start_step, total_steps), desc='Fine-tuning SapBERT V3 hard negatives')
    for step in progress:
        texts, labels, triplet_count = next(iterator)
        optimizer.zero_grad(set_to_none=True)
        embeddings = encode_train_batch(texts, tokenizer, model, device)
        supcon = supervised_contrastive_loss(embeddings, labels, TEMPERATURE)
        triplet = hard_negative_triplet_loss(embeddings, triplet_count, TRIPLET_MARGIN)
        loss = supcon + float(HARD_NEGATIVE_LOSS_WEIGHT) * triplet
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), float(MAX_GRAD_NORM))
        optimizer.step()
        scheduler.step()

        loss_value = float(loss.detach().cpu())
        supcon_value = float(supcon.detach().cpu())
        triplet_value = float(triplet.detach().cpu())
        running_loss = loss_value if running_loss is None else (0.95 * running_loss + 0.05 * loss_value)
        running_supcon = supcon_value if running_supcon is None else (0.95 * running_supcon + 0.05 * supcon_value)
        running_triplet = triplet_value if running_triplet is None else (0.95 * running_triplet + 0.05 * triplet_value)
        progress.set_postfix(loss=f'{running_loss:.4f}', triplet=f'{running_triplet:.4f}', best_mrr=f'{best_mrr:.4f}')

        if ((step + 1) % 100 == 0) or (step + 1 == total_steps):
            history.append({
                'step': int(step + 1),
                'loss': loss_value,
                'supcon_loss': supcon_value,
                'triplet_loss': triplet_value,
                'running_loss': float(running_loss),
                'running_supcon_loss': float(running_supcon),
                'running_triplet_loss': float(running_triplet),
                'elapsed_sec': float(time.perf_counter() - t0),
            })

        if ((step + 1) % int(VALIDATE_EVERY_STEPS) == 0) or (step + 1 == total_steps):
            metrics = evaluate_validation_retrieval(tokenizer, model, device)
            metrics_row = {
                'step': int(step + 1),
                'validation_MRR': metrics['MRR'],
                'validation_HitRate@1': metrics['HitRate@1'],
                'validation_HitRate@5': metrics['HitRate@5'],
                'validation_HitRate@10': metrics['HitRate@10'],
                'validation_median_rank': metrics['median_rank'],
            }
            history.append(metrics_row)
            write_training_history(history)
            print('Validation:', json.dumps(metrics_row, indent=2))
            if metrics['MRR'] > best_mrr:
                best_mrr = metrics['MRR']
                best_step = step + 1
                save_best_model(model, tokenizer, step + 1, metrics)

        if ((step + 1) % int(SAVE_EVERY_STEPS) == 0) or (step + 1 == total_steps):
            save_training_checkpoint(model, tokenizer, step + 1, history)

        del texts, labels, embeddings, supcon, triplet, loss
        if device.startswith('cuda') and (step + 1) % 25 == 0:
            torch.cuda.empty_cache()

    MODEL_LAST_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(MODEL_LAST_DIR)
    tokenizer.save_pretrained(MODEL_LAST_DIR)
    write_training_history(history)
    save_json(
        {
            'run_name': RUN_NAME,
            'start_model_source': START_MODEL_SOURCE,
            'best_model_dir': str(MODEL_BEST_DIR),
            'last_model_dir': str(MODEL_LAST_DIR),
            'best_step': int(best_step) if best_step is not None else None,
            'best_validation_MRR': float(best_mrr),
            'max_train_steps': int(MAX_TRAIN_STEPS),
            'elapsed_sec': float(time.perf_counter() - t0),
        },
        RESULTS_RUN_DIR / 'v3_hard_negative_metrics.json',
    )
    print('Saved best model:', MODEL_BEST_DIR)
    print('Saved last model:', MODEL_LAST_DIR)


## Training Curves and Stop/Continue Check


In [ ]:
import matplotlib.pyplot as plt

if not HISTORY_CSV_PATH.exists():
    print('No training history found yet:', HISTORY_CSV_PATH)
else:
    history_df = pd.read_csv(HISTORY_CSV_PATH)
    val_df = (
        history_df.dropna(subset=['validation_MRR'])
        .sort_values('step')
        .drop_duplicates(subset=['step'], keep='last')
        .reset_index(drop=True)
    )
    if val_df.empty:
        print('Training history exists, but it has no validation rows yet.')
    else:
        best_idx = val_df['validation_MRR'].idxmax()
        best_row = val_df.loc[best_idx]
        latest_row = val_df.iloc[-1]

        fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
        axes[0].plot(val_df['step'], val_df['validation_MRR'], marker='o', linewidth=2)
        axes[0].scatter([best_row['step']], [best_row['validation_MRR']], s=80, zorder=3)
        axes[0].set_title('Validation MRR')
        axes[0].set_xlabel('Training step')
        axes[0].set_ylabel('MRR')
        axes[0].grid(alpha=0.25)

        axes[1].plot(val_df['step'], val_df['validation_median_rank'], marker='o', linewidth=2, color='tab:orange')
        axes[1].scatter([latest_row['step']], [latest_row['validation_median_rank']], s=80, zorder=3, color='tab:orange')
        axes[1].set_title('Validation Median Rank')
        axes[1].set_xlabel('Training step')
        axes[1].set_ylabel('Median rank, lower is better')
        axes[1].grid(alpha=0.25)

        fig.suptitle(f'{RUN_NAME}: hard-negative validation trend', y=1.02)
        fig.tight_layout()
        plot_path = PLOTS_DIR / 'v3_validation_mrr_median_rank.png'
        fig.savefig(plot_path, dpi=200, bbox_inches='tight')
        plt.show()
        print('Saved plot:', plot_path)


In [ ]:
if not HISTORY_CSV_PATH.exists():
    print('No training history found yet.')
else:
    history_df = pd.read_csv(HISTORY_CSV_PATH)
    val_df = (
        history_df.dropna(subset=['validation_MRR'])
        .sort_values('step')
        .drop_duplicates(subset=['step'], keep='last')
        .reset_index(drop=True)
    )
    if len(val_df) < 2:
        print('Need at least two validation points before making a trend decision.')
    else:
        first = val_df.iloc[0]
        latest = val_df.iloc[-1]
        best = val_df.loc[val_df['validation_MRR'].idxmax()]
        recent = val_df.tail(min(4, len(val_df)))
        recent_mrr_delta = float(recent['validation_MRR'].iloc[-1] - recent['validation_MRR'].iloc[0])
        recent_rank_delta = float(recent['validation_median_rank'].iloc[-1] - recent['validation_median_rank'].iloc[0])

        print(f"Initial step {int(first['step'])}: MRR={first['validation_MRR']:.4f}, median_rank={first['validation_median_rank']:.1f}")
        print(f"Best step {int(best['step'])}: MRR={best['validation_MRR']:.4f}, median_rank={best['validation_median_rank']:.1f}")
        print(f"Latest step {int(latest['step'])}: MRR={latest['validation_MRR']:.4f}, median_rank={latest['validation_median_rank']:.1f}")
        print(f"Recent trend over {len(recent)} validation points: delta MRR={recent_mrr_delta:+.4f}, delta median_rank={recent_rank_delta:+.1f}")

        if best['validation_MRR'] > first['validation_MRR']:
            recommendation = 'Evaluate externally: V3 beat the starting checkpoint on internal MRR.'
        elif recent_mrr_delta > 0 and recent_rank_delta <= 0:
            recommendation = 'Continue training: recent MRR is improving and median rank is stable or better.'
        elif latest['validation_median_rank'] < first['validation_median_rank'] * 0.7:
            recommendation = 'Consider external top-k evaluation: median rank improved enough that candidate retrieval may be better even if MRR is flat.'
        else:
            recommendation = 'Stop or reduce LR: V3 is not clearly improving internal retrieval.'
        print('Recommendation:', recommendation)


## Final Validation Check


In [ ]:
if (MODEL_BEST_DIR / 'config.json').exists():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_BEST_DIR, use_fast=True)
    model = AutoModel.from_pretrained(MODEL_BEST_DIR).to(device).eval()
    final_validation = evaluate_validation_retrieval(tokenizer, model, device)
    save_json(final_validation, RESULTS_RUN_DIR / 'final_best_checkpoint_validation.json')
    print(json.dumps(final_validation, indent=2))
else:
    print('No best model found yet; run the training cell first.')


## Evaluate Externally

If V3 improves internal MRR or substantially improves median rank/HitRate@10, evaluate it with the external lab FAISS notebook:

```python
MODEL_DIR = REPO_DIR / 'results' / 'sapbert_full_114m_hard_negative_V3' / 'artifacts' / 'model_best'
RESULTS_RUN_DIR = REPO_DIR / 'results' / 'sapbert_full_114m_hard_negative_V3' / 'external_lab_unique_surface'
WORK_DIR = SCRATCH_ROOT / 'chem_synonymizer' / 'sapbert_full_114m_hard_negative_V3_external_lab_unique_surface'
REENCODE_CORPUS = True
REENCODE_QUERIES = True
REBUILD_FAISS_INDEX = True
```

Because this is a new checkpoint, do not reuse old corpus/query embeddings or old FAISS indexes from another model.
